# Análise Exploratória de Dados (EDA) — ESG Scoring
### Projeto ML I — CESAR School

---

## 1. Introdução

### 1.2. Contextualização

**Problema abordado:**  
O problema consiste em prever o nível de desempenho ESG (Environmental, Social e Governance) de empresas com base em suas características setoriais e de mercado. O target é a variável `total_level`, que classifica as empresas como **High** ou **Medium**, representando uma tarefa de **classificação binária**.

**Objetivo da solução:**  
Construir um modelo de machine learning capaz de classificar o nível ESG total de uma empresa a partir de atributos como indústria, bolsa de valores e moeda.

**Domínio da aplicação:**  
O dataset pertence ao domínio de **finanças sustentáveis e ESG (Environmental, Social, Governance)**. Os dados foram coletados de relatórios de ESG de empresas listadas nas bolsas NYSE e NASDAQ, com avaliação realizada em 2022.

**Importância da EDA para o projeto:**  
A análise exploratória é essencial para entender a estrutura dos dados, identificar redundâncias (como o data leakage potencial entre scores numéricos e grades categóricos), tratar valores ausentes, verificar o balanceamento das classes e selecionar as features mais informativas antes do treinamento do modelo. Sem uma EDA criteriosa, corre-se o risco de construir modelos com desempenho artificialmente inflado ou enviesado.

---
## Dicionário de Dados — Base Bronze

| Coluna | Descrição | Tipo | Valores possíveis / Exemplo |
|--------|-----------|------|-----------------------------||
| ticker | Código de negociação da empresa na bolsa | String | dis, gm, gww |
| name | Nome da empresa | String | Walt Disney Co |
| currency | Moeda de negociação | String | USD, CNY, EUR, CHF, BRL |
| exchange | Bolsa de valores onde a empresa está listada | String | NEW YORK STOCK EXCHANGE, INC. / NASDAQ NMS - GLOBAL MARKET |
| industry | Setor de atuação da empresa | String (48 categorias) | Technology, Biotechnology, Health Care |
| logo | URL do logotipo da empresa | String | https://static.finnhub.io/... |
| weburl | URL do site oficial da empresa | String | https://thewaltdisneycompany.com/ |
| environment_grade | Nota de desempenho ambiental (escala bond-rating) | Categórico (ordinal) | C, B, BB, BBB, A, AA → Ex: A |
| environment_level | Nível de desempenho ambiental | Categórico (ordinal) | Low, Medium, High, Excellent → Ex: High |
| social_grade | Nota de desempenho social (escala bond-rating) | Categórico (ordinal) | C, B, BB, BBB, A, AA → Ex: BB |
| social_level | Nível de desempenho social | Categórico (ordinal) | Low, Medium, High, Excellent → Ex: Medium |
| governance_grade | Nota de governança corporativa (escala bond-rating) | Categórico (ordinal) | C, B, BB, BBB, A, AA → Ex: BB |
| governance_level | Nível de governança corporativa | Categórico (ordinal) | Low, Medium, High, Excellent → Ex: Medium |
| environment_score | Score numérico do pilar ambiental | Inteiro | 200 a 719 → Ex: 510 |
| social_score | Score numérico do pilar social | Inteiro | 160 a 667 → Ex: 316 |
| governance_score | Score numérico do pilar de governança | Inteiro | 75 a 475 → Ex: 321 |
| total_score | Soma dos três sub-scores (E + S + G) | Inteiro | 600 a 1536 → Ex: 1147 |
| last_processing_date | Data de processamento dos dados | String | 19-04-2022 |
| total_grade | Nota ESG geral da empresa (escala bond-rating) | Categórico (ordinal) | B, BB, BBB, A → Ex: BBB |
| total_level | **TARGET** — Nível ESG geral da empresa | Categórico (ordinal) | Medium, High → Ex: High |
| cik | Código identificador da empresa na SEC (regulador americano) | Inteiro | 1744489 |

---
## 2. Inspeção dos Dados

### 2.1 Importação das bibliotecas

Bibliotecas utilizadas no projeto e suas respectivas finalidades:

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, ConfusionMatrixDisplay
)
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='Set2', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

print('Bibliotecas importadas com sucesso!')
print(f'Pandas: {pd.__version__} | NumPy: {np.__version__}')

### 2.2 Carregamento dos dados

Dataset está no formato parquet e para engine utilizaremos pyarrow, pois é mais rápido e amplamente suportado.

In [ ]:
DATA_PATH = '../data/bronze/esg_reporting_bronze.parquet'
df = pd.read_parquet(DATA_PATH, engine='pyarrow')

print(f'Dataset carregado: {df.shape[0]} linhas x {df.shape[1]} colunas')

### 2.3 Primeiras linhas e estrutura

In [ ]:
df.head()

### 2.4 Tipos das variáveis

In [ ]:
print(df.dtypes)
print(f'\nVariáveis numéricas : {df.select_dtypes(include=np.number).columns.tolist()}')
print(f'Variáveis categóricas: {df.select_dtypes(exclude=np.number).columns.tolist()}')

### 2.5 Dimensão da base

In [ ]:
print(f'Dimensão: {df.shape[0]} registros | {df.shape[1]} colunas')

### 2.6 Valores ausentes

In [ ]:
nulls = df.isnull().sum()
null_pct = (nulls / len(df) * 100).round(2)
null_df = pd.DataFrame({'Nulos': nulls, '% do Total': null_pct})
null_df = null_df[null_df['Nulos'] > 0].sort_values('% do Total', ascending=False)

print('=== Colunas com valores ausentes ===')
print(null_df)
print(f'\nTotal de colunas sem nulos: {(nulls == 0).sum()} de {df.shape[1]}')

### 2.7 Registros duplicados

In [ ]:
n_dup = df.duplicated().sum()
n_dup_ticker = df.duplicated(subset='ticker').sum()
print(f'Linhas duplicadas  : {n_dup}')
print(f'Tickers duplicados : {n_dup_ticker}')
print('Conclusão: Dataset sem duplicatas — cada linha representa uma empresa única.')

### 2.8 Cardinalidade das variáveis categóricas

In [ ]:
cat_cols = df.select_dtypes(exclude=np.number).columns
for col in cat_cols:
    print(f'{col:25s}: {df[col].nunique():4d} únicos')

---
## 3. Estatísticas Descritivas

### 3.1 Estatísticas dos scores numéricos

In [ ]:
score_cols = ['environment_score', 'social_score', 'governance_score', 'total_score']
desc = df[score_cols].describe().T
desc['cv_%'] = (desc['std'] / desc['mean'] * 100).round(2)
desc.round(2)

### 3.2 Interpretação das estatísticas descritivas

INTERPRETAÇÃO:
_________________________________________________________________________________
• environment_score: média=405, mediana=483. Média < mediana indica assimetria negativa — grupo expressivo de scores altos + cauda inferior.  
  CV=35.8%: alta dispersão relativa entre as empresas.

• social_score: média=292, mediana=302. Distribuição quase simétrica.  
  CV=19.5%: dispersão moderada — empresas mais homogêneas no pilar Social.

• governance_score: média=279, mediana=300. CV=16.9% — o pilar mais homogêneo.  
  Menor variação entre setores e empresas.

• total_score: média=976, mediana=1046. Intervalo [600, 1536].  
  É a soma exata dos 3 sub-scores — não usar como feature (leakage direto).

### 3.3 Frequência das variáveis categóricas relevantes

Colunas de interesse: `total_level`, `total_grade`, `environment_grade`, `social_grade`, `governance_grade` e `exchange`.

Essas variáveis foram selecionadas por terem relevância direta para o problema ESG: `total_level` é o target, `total_grade` é sua versão mais granular, as três grades (E, S, G) são importantes para entender o dado — porém conforme será demonstrado na seção 4.5, também introduzem leakage. A `exchange` é uma característica genuína da empresa, sem relação com a construção do target.

In [ ]:
cat_interest = ['total_level', 'total_grade', 'environment_grade',
                'social_grade', 'governance_grade', 'exchange']

for col in cat_interest:
    freq = df[col].value_counts()
    pct  = (freq / len(df) * 100).round(1)
    print(f'\n{col.upper()}')
    print(pd.concat([freq, pct], axis=1, keys=['qtd', '%']).to_string())

---
## 4. Tratamento dos Dados

### 4.1 Remoção de colunas irrelevantes para modelagem

Colunas a serem removidas e as devidas justificativas:
* **ticker, name** — identificadores únicos, sem poder preditivo
* **logo, weburl** — metadados visuais/web, irrelevantes para ESG
* **cik** — código regulatório SEC, apenas identificador
* **last_processing_date** — data de processamento, não atributo da empresa
* **total_score** — variável derivada diretamente dos sub-scores ESG, introduzindo leakage e redundância informacional, já que o target é construído a partir dessa mesma composição.
* **total_grade** — variável derivada diretamente de total_score, causando vazamento de informação e possível inflação artificial das métricas de desempenho.

In [ ]:
cols_to_drop = ['ticker', 'name', 'logo', 'weburl', 'cik',
                'last_processing_date', 'total_score', 'total_grade']

df_clean = df.drop(columns=cols_to_drop).copy()
print(f'Shape após remoção: {df_clean.shape}')

### 4.2 Tratamento de valores ausentes

A única feature que possui nulos é a `industry`, como há 47 categorias e poucos nulos (13 registros = 1.8%), preenchemos com `'Unknown'` para não perder registros válidos.

In [ ]:
print(f'Nulos em industry ANTES: {df_clean["industry"].isnull().sum()}')
df_clean['industry'] = df_clean['industry'].fillna('Unknown')
print(f'Nulos em industry DEPOIS: {df_clean["industry"].isnull().sum()}')
print(f'Total de nulos no dataset limpo: {df_clean.isnull().sum().sum()}')

### 4.3 Encoding ordinal das grades ESG

As grades possuem escala ordinal então podemos utilizar o OrdinalEncoder para preservar a hierarquia: **C < CCC < B < BB < BBB < A < AA**.

In [ ]:
GRADE_ORDER = ['C', 'CCC', 'B', 'BB', 'BBB', 'A', 'AA']
grade_cols  = ['environment_grade', 'social_grade', 'governance_grade']

oe = OrdinalEncoder(
    categories=[GRADE_ORDER] * 3,
    handle_unknown='use_encoded_value',
    unknown_value=-1
)
df_clean[grade_cols] = oe.fit_transform(df_clean[grade_cols])

print('Mapeamento ordinal (bond-rating scale):')
for i, g in enumerate(GRADE_ORDER):
    print(f'  {g:4s} → {i}')

df_clean['target'] = (df_clean['total_level'] == 'High').astype(int)
print(f'\nTarget: High=1 ({(df_clean["target"]==1).sum()}) | Medium=0 ({(df_clean["target"]==0).sum()})')

### 4.4 Encoding das demais variáveis

`exchange` e `currency` utilizaremos LabelEncoder, pois possuem baixa cardinalidade (2 e 5 valores). Para `industry` utilizamos Frequency Encoding — substitui cada categoria pela frequência com que ela aparece no dataset. Para os levels utilizamos OrdinalEncoder preservando a ordem Low < Medium < High < Excellent.

In [ ]:
le = LabelEncoder()
df_clean['exchange_enc'] = le.fit_transform(df_clean['exchange'])
df_clean['currency_enc'] = le.fit_transform(df_clean['currency'])

industry_freq = df_clean['industry'].value_counts().to_dict()
df_clean['industry_freq'] = df_clean['industry'].map(industry_freq)

LEVEL_ORDER = ['Low', 'Medium', 'High', 'Excellent']
level_cols  = ['environment_level', 'social_level', 'governance_level']
oe_level = OrdinalEncoder(
    categories=[LEVEL_ORDER] * 3,
    handle_unknown='use_encoded_value',
    unknown_value=-1
)
df_clean[level_cols] = oe_level.fit_transform(df_clean[level_cols])

print('Encodings aplicados:')
print('  exchange → exchange_enc  (LabelEncoder, 2 categorias)')
print('  currency → currency_enc  (LabelEncoder, 5 categorias)')
print('  industry → industry_freq (Frequency Encoding, 48 categorias)')
print('  *_level  → ordinal enc.  (Low=0, Medium=1, High=2, Excellent=3)')
print(f'\nShape final: {df_clean.shape}')

### 4.5 Definição dos dois cenários para análise de leakage

* **Cenário A** (ocorre leakage): Com sub-scores numéricos — os sub-scores possuem relação direta com a construção do target (`total_score = E + S + G`, e `total_level` é derivado do `total_score`), introduzindo vazamento de informação e potencial inflação artificial da performance do modelo.

* **Cenário B** (modelo real): **Apenas as 3 features genuinamente independentes do target**: `exchange_enc`, `currency_enc` e `industry_freq`. As grades ESG (`environment_grade`, `social_grade`, `governance_grade`) foram excluídas deste cenário pois, apesar de serem categóricas, são calculadas diretamente a partir dos sub-scores numéricos, que por sua vez determinam o `total_level`. Incluir as grades seria uma forma indireta de introduzir o mesmo leakage. As únicas features verdadeiramente externas ao processo de cálculo do ESG são a bolsa, a moeda e o setor da empresa.

In [ ]:
FEATURES_A = [
    'environment_score', 'social_score', 'governance_score',
    'environment_grade', 'social_grade', 'governance_grade',
    'environment_level', 'social_level', 'governance_level',
    'exchange_enc', 'currency_enc', 'industry_freq'
]

# Cenário B: apenas features genuinamente independentes do target
# Grades e levels foram removidos pois derivam dos sub-scores → leakage indireto
FEATURES_B = [
    'exchange_enc',
    'currency_enc',
    'industry_freq'
]

TARGET = 'target'
X_A = df_clean[FEATURES_A]
X_B = df_clean[FEATURES_B]
y   = df_clean[TARGET]

print(f'Cenário A: {len(FEATURES_A)} features (com sub-scores + grades + levels)')
print(f'Cenário B: {len(FEATURES_B)} features (apenas exchange, currency, industry)')
print(f'Distribuição do target: {y.value_counts().to_dict()}')

---
## 5. Análise Exploratória e Visualizações

### VIZ 1 — Distribuição das classes (target)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
counts = df['total_level'].value_counts()
colors = ['#2196F3', '#FF9800']

bars = axes[0].bar(counts.index, counts.values, color=colors, edgecolor='white', linewidth=1.2)
axes[0].set_title('Frequência Absoluta', fontweight='bold', pad=10)
axes[0].set_ylabel('Número de Empresas')
axes[0].set_xlabel('Classe (total_level)')
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 f'{val}\n({val/len(df)*100:.1f}%)', ha='center', va='bottom', fontsize=11)
axes[0].set_ylim(0, max(counts.values) * 1.2)

axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90, textprops={'fontsize': 12},
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Proporção das Classes', fontweight='bold', pad=10)

plt.suptitle('VIZ 1 — Distribuição do Target (total_level)', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

OBJETIVO: Verificar se há desbalanceamento de classes que exija tratamento.  
INTERPRETAÇÃO: 451 empresas High (62.5%) vs 271 Medium (37.5%).  
É possível identificar que na base só existe duas classes registradas: Medium e High. Ou seja, não há nenhuma empresa com maturidade ESG Low ou Excellent. Isso pode ser um problema para previsões futuras, já que a ausência total das classes Low e Excellent impede que o modelo aprenda padrões associados a esses níveis de maturidade ESG.   
Nota-se um desbalanceamento moderado (~1.7:1), o que exige duas medidas complementares já que High é aproximadamente 1.7 vezes maior que a de empresas Medium.

Apesar de não ser um desbalanceamento crítico, ele ainda pode influenciar o treinamento do modelo, fazendo com que a classe majoritária tenha maior peso nas previsões.

Por esse motivo, é válido utilizar `class_weight="balanced"` durante a modelagem, permitindo que o algoritmo atribua maior importância à classe minoritária (Medium) durante o processo de aprendizado.

Além disso, a métrica mais adequada para avaliação é o F1-macro, pois a acurácia simples poderia ser enganosa. Um modelo que sempre previsse a classe High já alcançaria cerca de 62.5% de acerto sem necessariamente aprender padrões relevantes dos dados.

O F1-macro penaliza erros em ambas as classes de forma mais equilibrada, tornando a avaliação mais confiável em cenários com desbalanceamento.

### VIZ 2 — Distribuição dos sub-scores por classe

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

score_info = [
    ('environment_score', 'Score Ambiental (E)', '#4CAF50'),
    ('social_score',      'Score Social (S)',     '#2196F3'),
    ('governance_score',  'Score Governança (G)', '#9C27B0'),
]

for ax, (col, title, color) in zip(axes, score_info):
    for level, alpha in [('High', 0.7), ('Medium', 0.5)]:
        subset = df[df['total_level'] == level][col]
        ax.hist(subset, bins=25, alpha=alpha, label=level,
                edgecolor='white', linewidth=0.5, density=True)
    ax.axvline(df[col].mean(), color='red', linestyle=':', linewidth=1.5, label='Média geral')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Score')
    ax.set_ylabel('Densidade')
    ax.legend(fontsize=9)

plt.suptitle('VIZ 2 — Distribuição dos Sub-Scores por Classe (High vs Medium)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

OBJETIVO: Identificar qual dos pilares ESG possui maior capacidade de diferenciar empresas Medium e High.

INTERPRETAÇÃO:

- Score Ambiental (E): High = 504 vs Medium = 239 → Delta = 265
- Score Social (S): High = 322 vs Medium = 243 → Delta = 78
- Score Governança (G): High = 300 vs Medium = 243 → Delta = 57

O pilar ambiental apresenta a maior diferença entre as classes, indicando que empresas classificadas como High tendem a possuir desempenho ambiental significativamente superior às empresas Medium.

Já os pilares Social e Governança apresentam diferenças bem menores, sugerindo menor capacidade de separação entre as classes.

**IMPACTO NA MODELAGEM:**

Esses scores confirmam o leakage: `total_level` é determinado pela soma dos sub-scores. Por isso nenhum deles pode ser usado no Cenário B — mesmo que indiretamente pelas grades.

### VIZ 3 — Matriz de Correlação

In [ ]:
num_cols_corr = ['environment_score', 'social_score', 'governance_score', 'total_score']
corr_df = df[num_cols_corr].copy()
corr_df['target'] = y.values
corr_matrix = corr_df.corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    corr_matrix,
    annot=True, fmt='.2f', cmap='RdYlGn',
    center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.8,
    cbar_kws={'shrink': 0.8, 'label': 'Correlação de Pearson'},
    ax=ax, annot_kws={'size': 12, 'weight': 'bold'}
)
ax.set_title('VIZ 3 — Matriz de Correlação (Scores + Target)', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

OBJETIVO: Detectar multicolinearidade e possíveis casos de data leakage entre as variáveis e o target.

INTERPRETAÇÃO:

A variável `total_score` apresenta correlação extremamente alta com o target, confirmando a existência de data leakage, já que o score ESG é construído diretamente a partir dos sub-scores.

Além disso, `environment_score` possui a maior correlação com o target, indicando ser o principal fator discriminativo entre as classes.

Também existe multicolinearidade entre os sub-scores ESG, algo esperado, pois todos contribuem para o cálculo do score final.

IMPACTO NA MODELAGEM:

No Cenário B, a remoção dos sub-scores, grades e levels derivados elimina o leakage e reduz os efeitos da multicolinearidade, permitindo uma avaliação mais realista do modelo.

### VIZ 4 — Pairplot dos scores numéricos

In [ ]:
pair_df = df[['environment_score', 'social_score',
               'governance_score', 'total_level']].copy()

g = sns.pairplot(
    pair_df, hue='total_level',
    palette={'High': '#2196F3', 'Medium': '#FF9800'},
    diag_kind='kde', plot_kws={'alpha': 0.5, 's': 25},
    diag_kws={'fill': True, 'alpha': 0.5}
)
g.figure.suptitle('VIZ 4 — Pairplot dos Sub-Scores ESG por Classe', y=1.02, fontsize=13, fontweight='bold')
plt.show()

OBJETIVO: Analisar relações bivariadas e a separabilidade visual entre as classes ESG.

INTERPRETAÇÃO:

O `environment_score` apresenta a melhor separação visual entre as classes, com pouca sobreposição nas distribuições, indicando alto poder discriminativo — mas isso se deve ao leakage, não a um padrão genuíno.

Já a combinação entre `social_score` e `governance_score` apresenta maior sobreposição entre as classes, sugerindo menor capacidade de diferenciação.

### VIZ 5 — Análise de Outliers — Boxplots

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 6))
score_cols_all = ['environment_score', 'social_score', 'governance_score', 'total_score']
labels_sc = ['Ambiental (E)', 'Social (S)', 'Governança (G)', 'Total ESG']
colors_sc = ['#4CAF50', '#2196F3', '#9C27B0', '#FF5722']

for ax, col, label, color in zip(axes, score_cols_all, labels_sc, colors_sc):
    bp = ax.boxplot(df[col], patch_artist=True, notch=True,
                    boxprops=dict(facecolor=color, alpha=0.7),
                    medianprops=dict(color='black', linewidth=2),
                    whiskerprops=dict(linewidth=1.5),
                    capprops=dict(linewidth=1.5),
                    flierprops=dict(marker='o', markerfacecolor=color, markersize=4, alpha=0.5))
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    n_out = ((df[col] < q1 - 1.5*iqr) | (df[col] > q3 + 1.5*iqr)).sum()
    ax.set_title(f'{label}\n({n_out} outliers IQR)', fontweight='bold', fontsize=10)
    ax.set_ylabel('Score')
    ax.set_xticks([])

plt.suptitle('VIZ 5 — Análise de Outliers por Pilar ESG (Boxplot com IQR)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

OBJETIVO: Detectar outliers utilizando a regra do IQR (1.5x) e avaliar possíveis assimetrias nas distribuições.

INTERPRETAÇÃO:

- Ambiental (E): IQR = 279 | nenhum outlier relevante identificado
- Social (S): IQR = 80 | presença de outliers
- Governança (G): IQR = 75 | presença de outliers

Os pilares Social e Governança apresentam valores extremos mais frequentes, indicando empresas com desempenho muito acima ou abaixo da média nesses indicadores. Esses outliers não aparentam ser erros de coleta, mas sim observações reais de empresas com comportamentos ESG mais extremos.

### VIZ 6 — Top 15 indústrias por Score ESG médio

In [ ]:
industry_stats = df.groupby('industry')['total_score'].agg(['mean', 'count'])
industry_stats = industry_stats[industry_stats['count'] >= 5].sort_values('mean', ascending=False)
top15 = industry_stats.head(15)
colors_grad = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(top15)))

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(top15.index, top15['mean'], color=colors_grad[::-1], edgecolor='white')
ax.axvline(df['total_score'].mean(), color='red', linestyle='--', linewidth=1.5,
           label=f'Média geral ({df["total_score"].mean():.0f})')
for bar, (idx, row) in zip(bars, top15.iterrows()):
    ax.text(bar.get_width() + 8, bar.get_y() + bar.get_height()/2,
            f'n={int(row["count"])}', va='center', fontsize=8, color='#555')
ax.set_xlabel('Score ESG Total Médio')
ax.set_title('VIZ 6 — Top 15 Indústrias por Score ESG Médio (n ≥ 5)', fontweight='bold', pad=12)
ax.legend()
ax.invert_yaxis()
plt.tight_layout()
plt.show()

OBJETIVO: Avaliar o poder preditivo genuíno da variável `industry` — sendo ela uma característica real da empresa, independente do cálculo do ESG.

INTERPRETAÇÃO:

Setores mais expostos à regulação ambiental, como Utilities, Chemicals e Materials, tendem a apresentar scores ESG mais elevados, possivelmente devido à maior pressão regulatória e necessidade de transparência.

Isso indica que o setor econômico influencia o comportamento ESG das empresas e pode contribuir para a diferenciação entre as classes — sendo uma feature legítima no Cenário B.

### VIZ 7 — Distribuição das grades por pilar

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
grade_order_display = ['C', 'CCC', 'B', 'BB', 'BBB', 'A', 'AA']
grade_raw_cols = ['environment_grade', 'social_grade', 'governance_grade']
titles_g = ['Grade Ambiental (E)', 'Grade Social (S)', 'Grade Governança (G)']
colors_g = ['#4CAF50', '#2196F3', '#9C27B0']

for ax, col, title, color in zip(axes, grade_raw_cols, titles_g, colors_g):
    counts_g = df[col].value_counts()
    order  = [g for g in grade_order_display if g in counts_g.index]
    values = [counts_g.get(g, 0) for g in order]
    bars = ax.bar(order, values, color=color, alpha=0.75, edgecolor='white', linewidth=0.8)
    for bar, val in zip(bars, values):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                    str(val), ha='center', va='bottom', fontsize=9)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Grade (Escala Bond-Rating)')
    ax.set_ylabel('Frequência')

plt.suptitle('VIZ 7 — Distribuição das Grades ESG por Pilar', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

OBJETIVO: Verificar concentração e variabilidade das grades em cada pilar ESG.

INTERPRETAÇÃO:

O pilar Ambiental apresenta maior variabilidade entre as empresas, com forte concentração nas grades A (321) e B (255), indicando uma distribuição mais polarizada entre desempenhos altos e intermediários.

Já o pilar Social é amplamente dominado pela grade BB (441), demonstrando baixa dispersão e comportamento mais homogêneo entre as empresas.

No pilar de Governança, as grades BB (434) e B (282) concentram a maior parte das observações, e nenhuma empresa atingiu a classificação AA.

**Nota:** Essa análise é relevante para entender os dados, mas as grades não serão usadas no Cenário B pois derivam dos sub-scores numéricos.

### VIZ 8 — Score ESG por Bolsa de Valores

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

df_exc = df.copy()
df_exc['exchange_short'] = df_exc['exchange'].apply(
    lambda x: 'NYSE' if 'NEW YORK' in x else 'NASDAQ')

data_by_exc = [df_exc[df_exc['exchange_short']=='NYSE']['total_score'],
               df_exc[df_exc['exchange_short']=='NASDAQ']['total_score']]

bp = axes[0].boxplot(data_by_exc, patch_artist=True, notch=True,
                      labels=['NYSE', 'NASDAQ'],
                      boxprops=dict(alpha=0.7),
                      medianprops=dict(color='black', linewidth=2.5))
for patch, color in zip(bp['boxes'], ['#FF5722', '#2196F3']):
    patch.set_facecolor(color)
axes[0].set_title('Score ESG Total por Bolsa', fontweight='bold')
axes[0].set_ylabel('Total Score')

prop = df_exc.groupby(['exchange_short', 'total_level']).size().unstack(fill_value=0)
prop_pct = prop.div(prop.sum(axis=1), axis=0) * 100
prop_pct.plot(kind='bar', ax=axes[1], color=['#2196F3', '#FF9800'], edgecolor='white', rot=0)
axes[1].set_title('Proporção High vs Medium por Bolsa', fontweight='bold')
axes[1].set_ylabel('% das Empresas')
axes[1].set_xlabel('Bolsa')
axes[1].legend(title='Nível ESG')
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter())

plt.suptitle('VIZ 8 — Desempenho ESG por Bolsa de Valores (NYSE vs NASDAQ)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

OBJETIVO: Verificar se a bolsa de listagem influencia o nível de maturidade ESG das empresas.

INTERPRETAÇÃO:

Empresas listadas na NYSE apresentam scores ESG ligeiramente superiores às listadas na NASDAQ. Apesar da diferença ser relativamente pequena, a variável `exchange` é uma feature **legítima** no Cenário B, pois é uma característica da empresa que não deriva do cálculo do ESG.

IMPACTO NA MODELAGEM:

Mesmo com impacto limitado quando analisada isoladamente, `exchange_enc` pode contribuir como feature contextual de baixo custo no Cenário B.

### VIZ 9 — Heatmap: Score médio por Indústria e Pilar

In [ ]:
top_industries = df.groupby('industry').size().nlargest(12).index
df_top = df[df['industry'].isin(top_industries)]
pivot = df_top.groupby('industry')[['environment_score', 'social_score', 'governance_score']].mean()
pivot.columns = ['Ambiental', 'Social', 'Governança']
pivot = pivot.sort_values('Ambiental', ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Score Médio'}, annot_kws={'size': 9})
ax.set_title('VIZ 9 — Score Médio por Indústria e Pilar ESG (Top 12 Setores)',
             fontweight='bold', pad=12)
ax.set_xlabel('Pilar ESG')
ax.set_ylabel('')
plt.xticks(rotation=0)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

OBJETIVO: Identificar padrões setoriais nos pilares ESG e possíveis interações entre setor e desempenho.

INTERPRETAÇÃO:

O pilar Ambiental apresenta a maior variação entre setores, evidenciada pelo maior contraste no heatmap. Isso sugere que fatores ambientais dependem fortemente do tipo de indústria e do nível de regulação ao qual as empresas estão expostas.

Já o pilar Social apresenta comportamento mais uniforme, indicando que empresas de diferentes setores tendem a possuir práticas sociais relativamente semelhantes.

IMPACTO NA MODELAGEM:

Essa variação por setor confirma que `industry` carrega informação preditiva real — sem depender do cálculo dos scores. Por isso ela é uma feature válida no Cenário B através do `industry_freq`.

### VIZ 10 — Violin Plot dos Sub-Scores por Classe

In [ ]:
df_melt = df[['environment_score', 'social_score',
               'governance_score', 'total_level']].melt(
    id_vars='total_level', var_name='Pilar', value_name='Score')
df_melt['Pilar'] = df_melt['Pilar'].map({
    'environment_score': 'Ambiental',
    'social_score': 'Social',
    'governance_score': 'Governança'
})

fig, ax = plt.subplots(figsize=(13, 6))
sns.violinplot(data=df_melt, x='Pilar', y='Score', hue='total_level',
               split=True, inner='quart',
               palette={'High': '#2196F3', 'Medium': '#FF9800'}, ax=ax)
ax.set_title('VIZ 10 — Violin Plot dos Sub-Scores por Nível ESG (High vs Medium)',
             fontweight='bold', pad=12)
ax.set_xlabel('Pilar ESG')
ax.set_ylabel('Score')
ax.legend(title='Nível Total')
plt.tight_layout()
plt.show()

OBJETIVO: Comparar formato, dispersão e separabilidade das distribuições ESG entre as classes.

INTERPRETAÇÃO:

O pilar Ambiental apresenta a separação mais clara entre empresas High e Medium — o que reforça o leakage: o target é basicamente determinado pelo ambiente score. Pilares Social e Governança têm sobreposição muito maior, evidenciando menor poder discriminativo quando isolados.

---
## 6. Seleção de Features com Biblioteca (Scikit-learn)

### Metodologia

Utilizamos **três classificadores** para avaliar a importância das features, aplicados nos dois cenários (A e B):

**1. Decision Tree** (`sklearn.tree.DecisionTreeClassifier`)**:**  
Gera importância de features baseada na redução de impureza Gini em cada split. Simples, interpretável e não requer normalização dos dados.

**2. XGBoost** (`xgboost.XGBClassifier`)**:**  
Gradient Boosting otimizado que combina várias árvores de decisão fracas. A importância de features é calculada pelo ganho médio de cada feature nos splits de todas as árvores.

**3. KNN** (`sklearn.neighbors.KNeighborsClassifier`)**:**  
O KNN não possui importância de features nativa. Por isso utilizamos **Permutation Importance**: embaralha cada feature aleatoriamente e mede o quanto o desempenho do modelo piora — quanto maior a queda, mais importante a feature.

**Por que esses três?**  
A combinação cobre abordagens distintas: Decision Tree e XGBoost são baseados em árvores (capturam não-linearidade), enquanto KNN com Permutation Importance avalia a feature pela sua contribuição real ao desempenho do modelo.

In [ ]:
# KNN requer dados normalizados (sensível à escala)
scaler = MinMaxScaler()
X_A_scaled = scaler.fit_transform(X_A)
X_B_scaled = scaler.fit_transform(X_B)

print('Dados normalizados para KNN!')
print('X_A_scaled shape:', X_A_scaled.shape)
print('X_B_scaled shape:', X_B_scaled.shape)

### 6.1 Seleção de features — Cenário A (COM sub-scores)

In [ ]:
# --- Decision Tree ---
dt_A = DecisionTreeClassifier(random_state=42, class_weight='balanced', max_depth=10)
dt_A.fit(X_A, y)
fi_dt_A = pd.Series(dt_A.feature_importances_, index=FEATURES_A).sort_values(ascending=False)

# --- XGBoost ---
xgb_A = XGBClassifier(random_state=42, eval_metric='logloss', verbosity=0,
                       scale_pos_weight=(y==0).sum()/(y==1).sum())
xgb_A.fit(X_A, y)
fi_xgb_A = pd.Series(xgb_A.feature_importances_, index=FEATURES_A).sort_values(ascending=False)

# --- KNN + Permutation Importance ---
knn_A = KNeighborsClassifier(n_neighbors=7)
knn_A.fit(X_A_scaled, y)
perm_A = permutation_importance(knn_A, X_A_scaled, y, n_repeats=10, random_state=42, n_jobs=-1)
fi_knn_A = pd.Series(perm_A.importances_mean, index=FEATURES_A).sort_values(ascending=False)

# --- Visualização ---
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

fi_dt_A.plot(kind='barh', ax=axes[0], color='#2196F3', alpha=0.8, edgecolor='white')
axes[0].set_title('Decision Tree — Cenário A\n(com sub-scores)', fontweight='bold')
axes[0].set_xlabel('Importância (Gini)')
axes[0].invert_yaxis()

fi_xgb_A.plot(kind='barh', ax=axes[1], color='#4CAF50', alpha=0.8, edgecolor='white')
axes[1].set_title('XGBoost — Cenário A\n(com sub-scores)', fontweight='bold')
axes[1].set_xlabel('Importância (Gain)')
axes[1].invert_yaxis()

fi_knn_A.plot(kind='barh', ax=axes[2], color='#FF9800', alpha=0.8, edgecolor='white')
axes[2].set_title('KNN Permutation — Cenário A\n(com sub-scores)', fontweight='bold')
axes[2].set_xlabel('Importância (Permutation)')
axes[2].invert_yaxis()

plt.suptitle('Seleção de Features — Cenário A (COM Sub-Scores = LEAKAGE CONFIRMADO)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('TOP FEATURES — CENÁRIO A:')
print('\nDecision Tree:')
print(fi_dt_A.round(4).to_string())
print('\nXGBoost:')
print(fi_xgb_A.round(4).to_string())
print('\nKNN (Permutation Importance):')
print(fi_knn_A.round(4).to_string())

### 6.2 Seleção de features — Cenário B (SEM sub-scores, SEM grades, SEM levels)

No Cenário B utilizamos apenas as 3 features genuinamente independentes do target: `exchange_enc`, `currency_enc` e `industry_freq`. A análise de importância aqui serve para confirmar se essas features carregam algum sinal preditivo real.

In [ ]:
# --- Decision Tree ---
dt_B = DecisionTreeClassifier(random_state=42, class_weight='balanced', max_depth=10)
dt_B.fit(X_B, y)
fi_dt_B = pd.Series(dt_B.feature_importances_, index=FEATURES_B).sort_values(ascending=False)

# --- XGBoost ---
xgb_B = XGBClassifier(random_state=42, eval_metric='logloss', verbosity=0,
                       scale_pos_weight=(y==0).sum()/(y==1).sum())
xgb_B.fit(X_B, y)
fi_xgb_B = pd.Series(xgb_B.feature_importances_, index=FEATURES_B).sort_values(ascending=False)

# --- KNN + Permutation Importance ---
knn_B = KNeighborsClassifier(n_neighbors=7)
knn_B.fit(X_B_scaled, y)
perm_B = permutation_importance(knn_B, X_B_scaled, y, n_repeats=10, random_state=42, n_jobs=-1)
fi_knn_B = pd.Series(perm_B.importances_mean, index=FEATURES_B).sort_values(ascending=False)

# --- Visualização ---
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

fi_dt_B.plot(kind='barh', ax=axes[0], color='#FF5722', alpha=0.8, edgecolor='white')
axes[0].set_title('Decision Tree — Cenário B\n(sem leakage)', fontweight='bold')
axes[0].set_xlabel('Importância (Gini)')
axes[0].invert_yaxis()

fi_xgb_B.plot(kind='barh', ax=axes[1], color='#9C27B0', alpha=0.8, edgecolor='white')
axes[1].set_title('XGBoost — Cenário B\n(sem leakage)', fontweight='bold')
axes[1].set_xlabel('Importância (Gain)')
axes[1].invert_yaxis()

fi_knn_B.plot(kind='barh', ax=axes[2], color='#009688', alpha=0.8, edgecolor='white')
axes[2].set_title('KNN Permutation — Cenário B\n(sem leakage)', fontweight='bold')
axes[2].set_xlabel('Importância (Permutation)')
axes[2].invert_yaxis()

plt.suptitle('Seleção de Features — Cenário B (exchange + currency + industry = Modelo Real)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('TOP FEATURES — CENÁRIO B:')
print('\nDecision Tree:')
print(fi_dt_B.round(4).to_string())
print('\nXGBoost:')
print(fi_xgb_B.round(4).to_string())
print('\nKNN (Permutation Importance):')
print(fi_knn_B.round(4).to_string())

### 6.3 Diagnóstico de leakage e features recomendadas

In [ ]:
print('=' * 65)
print('DIAGNÓSTICO DE DATA LEAKAGE')
print('=' * 65)
print()
print('CENÁRIO A (com sub-scores):')
print(f'  Top DT  : {fi_dt_A.index[0]} = {fi_dt_A.iloc[0]:.4f}')
print(f'  Top XGB : {fi_xgb_A.index[0]} = {fi_xgb_A.iloc[0]:.4f}')
print(f'  Top KNN : {fi_knn_A.index[0]} = {fi_knn_A.iloc[0]:.4f}')
print('  Sub-scores dominam os três métodos — LEAKAGE CONFIRMADO.')
print()
print('CENÁRIO B (exchange + currency + industry — sem leakage):')
print(f'  Top DT  : {fi_dt_B.index[0]} = {fi_dt_B.iloc[0]:.4f}')
print(f'  Top XGB : {fi_xgb_B.index[0]} = {fi_xgb_B.iloc[0]:.4f}')
print(f'  Top KNN : {fi_knn_B.index[0]} = {fi_knn_B.iloc[0]:.4f}')
print()
print('RANKING CONSOLIDADO — CENÁRIO B (média de posições):')
ranks = pd.DataFrame({
    'DT' : fi_dt_B.rank(ascending=False),
    'XGB': fi_xgb_B.rank(ascending=False),
    'KNN': fi_knn_B.rank(ascending=False)
})
ranks['rank_medio'] = ranks.mean(axis=1)
ranking_final = ranks.sort_values('rank_medio')

for i, (feat, row) in enumerate(ranking_final.iterrows(), 1):
    print(f'  {i}. {feat:20s} | DT={fi_dt_B.get(feat,0):.4f} | XGB={fi_xgb_B.get(feat,0):.4f} | KNN={fi_knn_B.get(feat,0):.4f}')

print()
print('JUSTIFICATIVA: As 3 features são genuinamente independentes do target.')
print('Evitar leakage garante desempenho realista em produção.')

---
## 7. Conclusões da EDA

### Principais Achados

1. **Desbalanceamento moderado de classes**: 62.5% High vs 37.5% Medium. Por não possuir outras classes como Low ou Excellent para o modelo aprender, acaba se tornando um problema crítico. Como só temos duas classes recomenda-se `class_weight='balanced'` e monitorar F1-macro na avaliação.

2. **Data leakage confirmado nos sub-scores e grades**: Os sub-scores numéricos são componentes diretos de `total_score`, que determina o target `total_level`. As grades ESG (environment_grade, social_grade, governance_grade) e os levels derivam diretamente dos sub-scores — incluí-las no Cenário B seria leakage indireto. O Cenário A serve apenas como benchmark para quantificar o efeito do leakage.

3. **Cenário B usa apenas 3 features**: `exchange_enc`, `currency_enc` e `industry_freq` — são as únicas variáveis genuinamente independentes do processo de cálculo do ESG. O desempenho esperado é 60-70%, muito abaixo dos 100% do Cenário A, mas representa um aprendizado real.

4. **Indústria é o preditor mais informativo no Cenário B**: Setores com maior regulação ambiental (Utilities, Energy, Chemicals) tendem a concentrar empresas High.

5. **Exchange com impacto marginal**: NYSE tem scores ligeiramente superiores; diferença pequena, mas útil como feature de contexto.

6. **Dataset de alta qualidade**: Sem duplicatas, apenas 13 nulos em `industry` (1.8%), tratados com preenchimento por 'Unknown'.

---

## 8. Treinamento dos Modelos — Cenário B

### Metodologia

O pipeline de treinamento segue as seguintes etapas:

1. **Holdout**: divisão em treino (80%) e teste (20%) com estratificação. O conjunto de teste é separado **antes de qualquer treinamento** e tratado como dados nunca vistos — não entra no GridSearch nem no KFold.

2. **GridSearchCV + StratifiedKFold**: aplicado **apenas no conjunto de treino**. Busca os melhores hiperparâmetros avaliando k=5 folds, mantendo a proporção das classes em cada fold.

3. **Retreino com melhores hiperparâmetros**: após identificar os melhores parâmetros, cada modelo é treinado novamente usando **todo o conjunto de treino** (sem folds).

4. **Avaliação final exclusivamente no teste**: as métricas reportadas — Accuracy, Precision, Recall, F1 e Confusion Matrix — são calculadas apenas sobre `X_test` / `y_test`.

### 8.1 Divisão holdout e configuração do pipeline

In [ ]:
# Holdout: 80% treino, 20% teste
# stratify=y garante a mesma proporção High/Medium em ambos os conjuntos
X_train, X_test, y_train, y_test = train_test_split(
    X_B, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'Treino : {X_train.shape[0]} amostras')
print(f'Teste  : {X_test.shape[0]} amostras')
print(f'Proporção High no treino : {y_train.mean():.1%}')
print(f'Proporção High no teste  : {y_test.mean():.1%}')
print('\nO conjunto de teste NÃO será usado durante o GridSearch nem no KFold.')

In [ ]:
# StratifiedKFold com k=5 — será aplicado somente no conjunto de treino
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Métricas de scoring para o GridSearchCV
# Usamos f1_weighted como critério de seleção (refit) por causa do desbalanceamento
SCORING = {
    'accuracy':  'accuracy',
    'precision': 'precision_weighted',
    'recall':    'recall_weighted',
    'f1':        'f1_weighted'
}

# Função para avaliar o modelo final no conjunto de teste
def avaliar_no_teste(modelo, nome):
    y_pred = modelo.predict(X_test)
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted')
    rec  = recall_score(y_test, y_pred, average='weighted')
    f1   = f1_score(y_test, y_pred, average='weighted')

    print(f'\n=== {nome} — Métricas no Conjunto de TESTE ===')
    print(f'  Accuracy:  {acc:.3f}')
    print(f'  Precision: {prec:.3f}')
    print(f'  Recall:    {rec:.3f}')
    print(f'  F1:        {f1:.3f}')
    print(f'\nRelatório por classe:')
    print(classification_report(y_test, y_pred, target_names=['Medium (0)', 'High (1)']))

    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred,
        display_labels=['Medium', 'High'],
        cmap='Blues'
    )
    plt.title(f'Matriz de Confusão — {nome} (Teste)')
    plt.show()

    return {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1}

print('Pipeline configurado.')

### 8.2 Modelo 1 — Decision Tree

A Decision Tree é interpretável e funciona bem com features categóricas encodadas. O `class_weight='balanced'` compensa o desbalanceamento atribuindo mais peso à classe minoritária (Medium).

In [ ]:
tree_params = {
    'criterion':         ['gini', 'entropy'],
    'max_depth':         [3, 5, 7, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf':  [1, 2, 4],
}

# GridSearchCV aplicado SOMENTE no treino (X_train, y_train)
tree_grid = GridSearchCV(
    estimator=DecisionTreeClassifier(random_state=42, class_weight='balanced'),
    param_grid=tree_params,
    scoring=SCORING,
    refit='f1',
    cv=skf,
    n_jobs=-1,
    verbose=1
)
tree_grid.fit(X_train, y_train)

print(f'\nMelhores parâmetros: {tree_grid.best_params_}')
best_idx = tree_grid.best_index_
print(f'F1 médio no CV (treino): {tree_grid.cv_results_["mean_test_f1"][best_idx]:.3f}')

In [ ]:
# O best_estimator_ já foi retreinado com todos os dados de treino após o GridSearch
# (sklearn faz isso automaticamente quando refit=True)
best_tree = tree_grid.best_estimator_
tree_metrics = avaliar_no_teste(best_tree, 'Decision Tree')

### 8.3 Modelo 2 — KNN

O KNN é sensível à escala das features, por isso normalizamos os dados antes do treino. O `fit` do scaler é feito **somente no treino** e depois aplicado ao teste — isso evita que informação do teste vaze para o treino.

In [ ]:
# Normalização — fit SOMENTE no treino, transform em treino e teste
scaler_knn = MinMaxScaler()
X_train_scaled = scaler_knn.fit_transform(X_train)
X_test_scaled  = scaler_knn.transform(X_test)   # usa o mesmo scaler, sem refitar

knn_params = {
    'n_neighbors': [3, 5, 7, 9, 11, 15],
    'weights':     ['uniform', 'distance'],
    'metric':      ['euclidean', 'manhattan'],
}

knn_grid = GridSearchCV(
    estimator=KNeighborsClassifier(),
    param_grid=knn_params,
    scoring=SCORING,
    refit='f1',
    cv=skf,
    n_jobs=-1,
    verbose=1
)
knn_grid.fit(X_train_scaled, y_train)

print(f'\nMelhores parâmetros: {knn_grid.best_params_}')
best_idx = knn_grid.best_index_
print(f'F1 médio no CV (treino): {knn_grid.cv_results_["mean_test_f1"][best_idx]:.3f}')

In [ ]:
# Para o KNN avaliamos nos dados escalados do teste
best_knn = knn_grid.best_estimator_
y_pred_knn = best_knn.predict(X_test_scaled)

acc  = accuracy_score(y_test, y_pred_knn)
prec = precision_score(y_test, y_pred_knn, average='weighted')
rec  = recall_score(y_test, y_pred_knn, average='weighted')
f1   = f1_score(y_test, y_pred_knn, average='weighted')
knn_metrics = {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1}

print('\n=== KNN — Métricas no Conjunto de TESTE ===')
print(f'  Accuracy:  {acc:.3f}')
print(f'  Precision: {prec:.3f}')
print(f'  Recall:    {rec:.3f}')
print(f'  F1:        {f1:.3f}')
print('\nRelatório por classe:')
print(classification_report(y_test, y_pred_knn, target_names=['Medium (0)', 'High (1)']))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_knn,
    display_labels=['Medium', 'High'],
    cmap='Blues'
)
plt.title('Matriz de Confusão — KNN (Teste)')
plt.show()

### 8.4 Modelo 3 — XGBoost

O XGBoost constrói várias árvores em sequência, onde cada nova tenta corrigir os erros da anterior. Usamos `scale_pos_weight` para compensar o desbalanceamento (equivalente ao `class_weight='balanced'` de outros modelos).

In [ ]:
# scale_pos_weight = qtd negativos / qtd positivos — compensa o desbalanceamento
scale_pw = (y_train == 0).sum() / (y_train == 1).sum()

xgb_params = {
    'n_estimators':     [50, 100, 200],
    'learning_rate':    [0.01, 0.05, 0.1],
    'max_depth':        [2, 3, 4],
    'min_child_weight': [1, 3],
    'subsample':        [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
}

xgb_grid = GridSearchCV(
    estimator=XGBClassifier(
        eval_metric='logloss',
        scale_pos_weight=scale_pw,
        random_state=42,
        verbosity=0
    ),
    param_grid=xgb_params,
    scoring=SCORING,
    refit='f1',
    cv=skf,
    n_jobs=-1,
    verbose=1
)
xgb_grid.fit(X_train, y_train)

print(f'\nMelhores parâmetros: {xgb_grid.best_params_}')
best_idx = xgb_grid.best_index_
print(f'F1 médio no CV (treino): {xgb_grid.cv_results_["mean_test_f1"][best_idx]:.3f}')

In [ ]:
best_xgb = xgb_grid.best_estimator_
xgb_metrics = avaliar_no_teste(best_xgb, 'XGBoost')

### 8.5 Comparação final dos 3 modelos no conjunto de teste

Todas as métricas abaixo foram calculadas **exclusivamente no conjunto de teste**, que não participou do treinamento nem da validação cruzada.

In [ ]:
resultados = pd.DataFrame({
    'Decision Tree': tree_metrics,
    'KNN':           knn_metrics,
    'XGBoost':       xgb_metrics,
}).T.round(3)

print('=== Comparação Final — Métricas no Conjunto de TESTE ===')
print(resultados.to_string())
print(f'\nBaseline (sempre High): ~{y_test.mean():.1%}')

resultados.plot(
    kind='bar', figsize=(12, 5), ylim=(0, 1), rot=0,
    color=['steelblue', 'tomato', 'seagreen', 'gold']
)
plt.title('Comparação de Métricas no Teste — Decision Tree vs KNN vs XGBoost (Cenário B)')
plt.ylabel('Score')
plt.axhline(y_test.mean(), color='gray', linestyle='--', label=f'Baseline ({y_test.mean():.1%})')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

melhor = resultados['f1'].idxmax()
print(f'\nMelhor modelo pelo F1 no teste: {melhor} ({resultados.loc[melhor, "f1"]:.3f})')